# UGTM Plagiarism AI — Baseline Error Analysis & Taxonomy (CP-03.5)

Phân tích chuyên sâu 5 cấu hình Baseline (B0, B1, B2, B3, B4) trên tập kiểm thử phát triển `data/benchmark_v1/val.csv` (120 cặp, 10 document families, 7 lát cắt).

## Mục tiêu nghiên cứu:
1. **RQ1:** Đánh giá vai trò của Cross-Encoder Reranking trong việc dập tắt False Alarms trên các bài cùng chủ đề (`hard_negatives`).
2. **RQ2:** So sánh năng lực của Lexical vs Dense vs Hybrid Retrieval trên các lát cắt dịch thuật và diễn giải lại (`cross_lingual_vi_en`, `paraphrase`, `summarization`).
3. **Error Taxonomy:** Phân bổ các mẫu sai vào 7 nhóm lỗi kỹ thuật.

In [ ]:
import json
import csv
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..")
REPORTS_DIR = BASE_DIR / "reports" / "baseline_reports"
PREDICTIONS_DIR = BASE_DIR / "data" / "baseline_predictions"
VAL_CSV = BASE_DIR / "data" / "benchmark_v1" / "val.csv"

baselines = ["b0", "b1", "b2", "b3", "b4"]
reports = {}
for b in baselines:
    rep_path = REPORTS_DIR / f"report_{b}.json"
    if rep_path.is_file():
        reports[b.upper()] = json.loads(rep_path.read_text(encoding="utf-8"))

print(f"Loaded {len(reports)} baseline reports.")

## 1. Bảng So Sánh Chỉ Số Chính Giữa B0 – B4

In [ ]:
rows = []
for b, rep in reports.items():
    c = rep["classification"]
    r = rep["retrieval"]
    fa = rep["false_alarms"]
    rows.append({
        "Baseline": b,
        "Source Recall@10": r.get("recall@10", 0.0),
        "Source Recall@50": r.get("recall@50", 0.0),
        "Pair Precision": c.get("precision", 0.0),
        "Pair Recall": c.get("recall", 0.0),
        "Pair F1-score": c.get("f1", 0.0),
        "95% CI": f"[{c['bootstrap_ci_95']['ci_lower']:.3f}, {c['bootstrap_ci_95']['ci_upper']:.3f}]",
        "PR-AUC": c.get("pr_auc", 0.0),
        "Brier Score": c.get("brier_score", 0.0),
        "False Alarms/Doc": fa.get("false_alarms_per_doc", 0.0),
    })

df_summary = pd.DataFrame(rows)
df_summary

## 2. Cây Phân Loại Lỗi (Error Taxonomy Breakdown)

In [ ]:
err_rows = []
for b, rep in reports.items():
    err = rep.get("error_taxonomy", {})
    err["Baseline"] = b
    err_rows.append(err)

df_errors = pd.DataFrame(err_rows).set_index("Baseline")
df_errors

## 3. Phân Rã Hiệu Năng F1 Theo 7 Lát Cắt (Slice Performance)

In [ ]:
slice_data = {}
for b, rep in reports.items():
    slices = rep.get("slice_breakdown", {})
    slice_data[b] = {sl: metrics.get("f1", 0.0) for sl, metrics in slices.items()}

df_slices = pd.DataFrame(slice_data)
df_slices

## 4. Kết Luận Khoa Học & Định Hướng Nâng Cấp CP-04

1. **B1 (Lexical-only)** không thể phát hiện đạo văn dịch thuật (`cross_lingual_vi_en`) và tóm tắt (`summarization`).
2. **B2 (Dense-only)** và **B3 (Hybrid RRF)** có Recall cao nhưng sinh ra số lượng cảnh báo giả rất lớn trên các bài cùng chủ đề (`hard_negatives`), khiến tỷ lệ False Alarm lên tới `0.6923` cảnh báo/bài.
3. **B4 (Hybrid RRF + Reranker)** dập tắt hoàn toàn False Alarms trên `hard_negatives` (nhờ Cross-Encoder token interaction), đồng thời duy trì Recall 100% trên cả 5 lát cắt dương tính.
4. **Kế hoạch cho CP-04:** Chuyển sang tối ưu hóa suy luận (K-sweep, latency profiling) và xây dựng bộ căn gióng động mức câu (Sentence Alignment) với IoU cao.